In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# === BOOTSTRAP: RUN THIS FIRST ===
import os, sys
from pathlib import Path
REPO_PATH = Path("/content/drive/MyDrive/MaintainAI/code")
os.chdir(REPO_PATH)
sys.path.insert(0, str(REPO_PATH))
print("ROOT:", REPO_PATH)
print("Exists:", REPO_PATH.exists())
print("src exists:", (REPO_PATH / "src").exists())

ROOT: /content/drive/MyDrive/MaintainAI/code
Exists: True
src exists: True


# 04 — Predictive model (Ridge baseline vs XGBoost)
Selection on VAL engines; final numbers on untouched NASA test trajectories.

In [3]:
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

from src.data_cmapss import load_fd001, add_rul, add_capped_rul, engine_split, SENSOR_COLS
from src.features import build_features, feature_columns, fit_scaler, apply_scaler
from src.predictive import (
    train_ridge, train_xgb_regressor, train_xgb_classifier,
    regression_metrics, classifier_metrics, tune_threshold,
    nasa_score, MODEL_VERSION, RUL_CAP, HORIZON
)

# Load processed features
feat = pd.read_parquet('data/processed/train_features.parquet')
schema = json.load(open('models/predictive/feature_schema.json'))
cols, tr_units, va_units = schema['feature_columns'], schema['train_units'], schema['val_units']
scaler = joblib.load('models/predictive/scaler_fd001.joblib')

# Split
tr = feat[feat['unit'].isin(tr_units)].copy()
va = feat[feat['unit'].isin(va_units)].copy()

# Scale
tr_scaled = apply_scaler(tr, cols, scaler)
va_scaled = apply_scaler(va, cols, scaler)

# Targets
y_rul_tr = tr_scaled['rul_capped'].to_numpy()
y_rul_va = va_scaled['rul_capped'].to_numpy()
y_clf_tr = (tr_scaled['rul'] <= HORIZON).astype(int).to_numpy()
y_clf_va = (va_scaled['rul'] <= HORIZON).astype(int).to_numpy()

X_tr = tr_scaled[cols].to_numpy()
X_va = va_scaled[cols].to_numpy()

print(f'Train: {X_tr.shape[0]} samples | Val: {X_va.shape[0]} samples')
print(f'RUL range train: [{y_rul_tr.min()}, {y_rul_tr.max()}] | Val: [{y_rul_va.min()}, {y_rul_va.max()}]')
print(f'Class balance train: {y_clf_tr.mean():.3f} | Val: {y_clf_va.mean():.3f}')

Train: 16342 samples | Val: 4289 samples
RUL range train: [0, 125] | Val: [0, 125]
Class balance train: 0.152 | Val: 0.145


In [4]:
# Train Ridge baseline (RUL regression)
print('Training Ridge baseline...')
ridge = train_ridge(X_tr, y_rul_tr)
ridge_rul_tr = ridge.predict(X_tr)
ridge_rul_va = ridge.predict(X_va)
print('Ridge Train:', regression_metrics(y_rul_tr, ridge_rul_tr))
print('Ridge Val:', regression_metrics(y_rul_va, ridge_rul_va))

Training Ridge baseline...
Ridge Train: {'rmse': 16.793953105013234, 'mae': 13.685921756211293, 'nasa_score': 77465.4447969367, 'n': 16342}
Ridge Val: {'rmse': 17.703087513115282, 'mae': 14.409801516525825, 'nasa_score': 22761.087636731605, 'n': 4289}


In [5]:
# Train XGBoost Regressor (RUL)
print('Training XGBoost Regressor...')
xgb_reg = train_xgb_regressor(X_tr, y_rul_tr)
xgb_rul_tr = xgb_reg.predict(X_tr)
xgb_rul_va = xgb_reg.predict(X_va)
print('XGB Train:', regression_metrics(y_rul_tr, xgb_rul_tr))
print('XGB Val:', regression_metrics(y_rul_va, xgb_rul_va))

Training XGBoost Regressor...
XGB Train: {'rmse': 4.648928450248494, 'mae': 3.2691109950466486, 'nasa_score': 6882.149605659863, 'n': 16342}
XGB Val: {'rmse': 14.493570977525, 'mae': 10.389489413947597, 'nasa_score': 15949.65886543583, 'n': 4289}


In [6]:
# Train XGBoost Classifier (risk within HORIZON cycles)
print('Training XGBoost Classifier...')
xgb_clf = train_xgb_classifier(X_tr, y_clf_tr)
clf_prob_tr = xgb_clf.predict_proba(X_tr)[:, 1]
clf_prob_va = xgb_clf.predict_proba(X_va)[:, 1]

# Tune threshold on VAL only
best_thresh = tune_threshold(y_clf_va, clf_prob_va)
print(f'Best threshold (val): {best_thresh:.3f}')
print('XGB Classifier Val:', classifier_metrics(y_clf_va, clf_prob_va, best_thresh))

Training XGBoost Classifier...
Best threshold (val): 0.700
XGB Classifier Val: {'threshold': 0.7000000000000002, 'precision': 0.9357021996615905, 'recall': 0.8919354838709678, 'f1': 0.9132947976878613, 'roc_auc': 0.9949467640826805, 'n': 4289, 'n_positive': 620}


In [7]:
# SELECTION: XGBoost beats Ridge on Val -> choose XGBoost
print('\n=== MODEL SELECTION ===')
ridge_val_rmse = regression_metrics(y_rul_va, ridge_rul_va)['rmse']
xgb_val_rmse = regression_metrics(y_rul_va, xgb_rul_va)['rmse']
print(f'Ridge Val RMSE: {ridge_val_rmse:.2f}')
print(f'XGB Val RMSE: {xgb_val_rmse:.2f}')
print(f'Selected: {"XGBoost" if xgb_val_rmse < ridge_val_rmse else "Ridge"}')

# Save selected models
joblib.dump(xgb_reg, 'models/predictive/rul_model.joblib')
joblib.dump(xgb_clf, 'models/predictive/risk_model.joblib')

# Save metadata
metadata = {
    'model_version': MODEL_VERSION,
    'rul_cap': RUL_CAP,
    'horizon': HORIZON,
    'decision_threshold': float(best_thresh),
    'feature_window': 30,
    'train_units': tr_units,
    'val_units': va_units
}
with open('models/predictive/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=1)

print('✓ Saved: models/predictive/rul_model.joblib')
print('✓ Saved: models/predictive/risk_model.joblib')
print('✓ Saved: models/predictive/metadata.json')


=== MODEL SELECTION ===
Ridge Val RMSE: 17.70
XGB Val RMSE: 14.49
Selected: XGBoost
✓ Saved: models/predictive/rul_model.joblib
✓ Saved: models/predictive/risk_model.joblib
✓ Saved: models/predictive/metadata.json


In [8]:
# HELD-OUT EVALUATION on NASA test trajectories (NEVER used for training/selection)
print('\n=== HELD-OUT TEST EVALUATION (NASA FD001) ===')

tr_raw, te_raw, rul_true = load_fd001('CMAPSSData')

# Build test features (same pipeline)
te_feat = build_features(te_raw)
# Compute true RUL for test: last_cycle + RUL_file - current_cycle
last_cycle = te_raw.groupby('unit')['cycle'].max()
te_feat = te_feat.merge(last_cycle.rename('last_cycle'), on='unit')
te_feat['rul'] = te_feat['last_cycle'] + rul_true.values[te_feat['unit'].astype(int) - 1] - te_feat['cycle']
te_feat['rul_capped'] = te_feat['rul'].clip(upper=RUL_CAP)

# Scale test features
te_scaled = apply_scaler(te_feat, cols, scaler)
X_te = te_scaled[cols].to_numpy()
y_rul_te = te_scaled['rul'].to_numpy()  # UNCAPPED official RUL
y_rul_te_capped = te_scaled['rul_capped'].to_numpy()
y_clf_te = (te_scaled['rul'] <= HORIZON).astype(int).to_numpy()

# Predict
rul_pred = np.maximum(0.0, xgb_reg.predict(X_te))
clf_prob_te = xgb_clf.predict_proba(X_te)[:, 1]

# Regression metrics (on UNCAPPED official RUL)
rul_metrics = regression_metrics(y_rul_te, rul_pred)
print('RUL (uncapped official):', rul_metrics)

# Classification metrics
clf_metrics = classifier_metrics(y_clf_te, clf_prob_te, best_thresh)
print(f'Risk (RUL≤{HORIZON}):', clf_metrics)

# Save test metrics
test_metrics = {
    'test_rul': rul_metrics,
    'test_clf_30cycle': clf_metrics
}
with open('models/predictive/metrics.json', 'w') as f:
    json.dump(test_metrics, f, indent=1)
print('\n✓ Saved: models/predictive/metrics.json')


=== HELD-OUT TEST EVALUATION (NASA FD001) ===
RUL (uncapped official): {'rmse': 56.93375524574018, 'mae': 41.258674024625314, 'nasa_score': 517622601.15025246, 'n': 13096}
Risk (RUL≤30): {'threshold': 0.7000000000000002, 'precision': 0.855595667870036, 'recall': 0.713855421686747, 'f1': 0.7783251231527094, 'roc_auc': 0.9973756668793632, 'n': 13096, 'n_positive': 332}

✓ Saved: models/predictive/metrics.json


In [9]:
# Verify PredictionService works end-to-end
from src.predictive import PredictionService

svc = PredictionService('models/predictive')
# Test on a fresh engine (early life)
raw_cols = ['unit', 'cycle', 'op1', 'op2', 'op3'] + SENSOR_COLS
fresh = te_raw[te_raw['unit'] == 1].head(10)[raw_cols]
aged = te_raw[te_raw['unit'] == 1].tail(30)[raw_cols]

fresh_pred = svc.predict(fresh)
aged_pred = svc.predict(aged)

print('Fresh engine (early cycles):', {
    'failure_prob': f"{fresh_pred['failure_probability']:.3f}",
    'health': fresh_pred['health_state'],
    'RUL': fresh_pred['rul_cycles']
})
print('Aged engine (late cycles):', {
    'failure_prob': f"{aged_pred['failure_probability']:.3f}",
    'health': aged_pred['health_state'],
    'RUL': aged_pred['rul_cycles']
})
print(f"\nAged failure_prob >= Fresh: {aged_pred['failure_probability'] >= fresh_pred['failure_probability']}")

Fresh engine (early cycles): {'failure_prob': '0.000', 'health': 'HEALTHY', 'RUL': 127}
Aged engine (late cycles): {'failure_prob': '0.000', 'health': 'HEALTHY', 'RUL': 122}

Aged failure_prob >= Fresh: True
